# 02 — Face Alignment + Mouth Extraction (96×96)

Notebook นี้ต่อจากผลของ `face_detection_yolov11_gpu.ipynb` โดยตรง

**Input**
- `dataset/raw_videos/video001.webm`
- `dataset/yolo_face_detection/video001/detection.csv`

**Output**
- `dataset/mouth_crops/video001/track_000/*.jpg`
- `dataset/mouth_crops/video001/track_001/*.jpg`
- `dataset/mouth_crops/video001/mouth_metadata.csv`
- `dataset/mouth_crops/video001/preview/*.jpg`

หลักการ:
1. ใช้ bounding box จาก YOLO Face เดิม
2. ใช้ `face-alignment` (FAN, PyTorch CUDA) หา 68 facial landmarks
3. Align ใบหน้าด้วยตาซ้าย + ตาขวา + จุดกึ่งกลางปาก
4. Crop ปากจาก aligned face แบบ fixed-size
5. Resize เป็น 96×96
6. แยกเก็บตาม `track_id`

> จุดริมฝีปากของ landmark 68 จุดคือ index `48:68`

## Cell 1 — ตรวจ GPU / Python

In [ ]:
import sys
import torch

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)
print("Python :", sys.executable)
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook นี้ควรใช้ PyTorch CUDA")

print("GPU    :", torch.cuda.get_device_name(0))

## Cell 2 — ติดตั้ง dependency

`face-alignment` เวอร์ชันใหม่ใช้ PyTorch และรองรับการส่ง bounding box ที่ตรวจไว้แล้วเข้าไปได้  
ครั้งแรกที่สร้างโมเดล อาจมีการดาวน์โหลด weight ของ FAN

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "face-alignment==1.5.0",
    "opencv-python",
    "numpy",
    "pandas",
    "matplotlib",
    "pillow"
])

print("Dependencies ready")

## Cell 3 — ตั้งค่า Path

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")

VIDEO_ID = "video001"

VIDEO_PATH = PROJECT_ROOT / "dataset" / "raw_videos" / f"{VIDEO_ID}.webm"
DETECTION_CSV = PROJECT_ROOT / "dataset" / "yolo_face_detection" / VIDEO_ID / "detection.csv"

OUTPUT_ROOT = PROJECT_ROOT / "dataset" / "mouth_crops" / VIDEO_ID
PREVIEW_DIR = OUTPUT_ROOT / "preview"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("VIDEO_PATH    :", VIDEO_PATH)
print("DETECTION_CSV :", DETECTION_CSV)
print("OUTPUT_ROOT   :", OUTPUT_ROOT)

print("\\nExists:")
print("Video:", VIDEO_PATH.exists())
print("CSV  :", DETECTION_CSV.exists())

if not VIDEO_PATH.exists():
    raise FileNotFoundError(VIDEO_PATH)

if not DETECTION_CSV.exists():
    raise FileNotFoundError(DETECTION_CSV)

## Cell 4 — อ่าน detection.csv และตรวจ Track

In [ ]:
import pandas as pd

det_df = pd.read_csv(DETECTION_CSV)

required_columns = {
    "frame_index", "track_id",
    "x1", "y1", "x2", "y2",
    "det_score"
}

missing = required_columns - set(det_df.columns)

if missing:
    raise ValueError(f"CSV ขาด columns: {missing}")

det_df = det_df.sort_values(
    ["frame_index", "track_id"]
).reset_index(drop=True)

print("Rows       :", len(det_df))
print("Frames     :", det_df["frame_index"].nunique())
print("Track IDs  :", sorted(det_df["track_id"].unique().tolist()))

display(det_df.head(10))

## Cell 5 — โหลด 68-point Face Alignment Model

เรา **ไม่ใช้ face detector ของ FAN เป็นตัวหลัก**  
YOLO Face ยังคงเป็นตัวตรวจหน้า ส่วน FAN ใช้หา landmarks จาก bounding box ที่มีอยู่แล้ว

ตั้ง `compile=False` เพื่อให้เริ่มทำงานง่ายและ debug ง่ายก่อน

In [ ]:
import face_alignment

fa = face_alignment.FaceAlignment(
    face_alignment.LandmarksType.TWO_D,
    device="cuda",
    flip_input=False,
    compile=False
)

print("Face Alignment ready")

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--force-reinstall",
    "numpy==2.3.3",
    "numba==0.62.1",
    "llvmlite==0.45.1"
])

In [ ]:
import sys
import numpy as np
import torch
import face_alignment

print("Python :", sys.executable)
print("Version:", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Face Alignment:", face_alignment.__version__)

In [ ]:
import sys

print("Python:", sys.executable)
print("Version:", sys.version)

In [ ]:
import sys
import subprocess

print("Installing into:")
print(sys.executable)

result = subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "face-alignment==1.5.0"
])

print("Return code:", result.returncode)

In [ ]:
import sys
import face_alignment

print("Python:", sys.executable)
print("Face Alignment:", face_alignment.__version__)

In [ ]:
import torch
import face_alignment

print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

print("Face Alignment:", face_alignment.__version__)

In [ ]:
import face_alignment

fa = face_alignment.FaceAlignment(
    face_alignment.LandmarksType.TWO_D,
    device="cuda",
    flip_input=False,
    compile=False
)

print("Face Alignment ready")

In [ ]:
import numpy as np
import numba
import llvmlite

print("NumPy   :", np.__version__)
print("Numba   :", numba.__version__)
print("llvmlite:", llvmlite.__version__)

## Cell 6 — ฟังก์ชัน Align Face + Crop Mouth

In [ ]:
import cv2
import numpy as np
import math

ALIGNED_SIZE = 256
MOUTH_OUTPUT_SIZE = 96

# จุดอ้างอิงปลายทางของใบหน้าที่ align แล้ว
DST_LEFT_EYE  = np.array([76.0, 92.0], dtype=np.float32)
DST_RIGHT_EYE = np.array([180.0, 92.0], dtype=np.float32)
DST_MOUTH     = np.array([128.0, 174.0], dtype=np.float32)

# Crop แบบ fixed-size บน aligned face
MOUTH_CROP_SIZE = 104


def transform_points(points, matrix):
    points = np.asarray(points, dtype=np.float32)
    ones = np.ones((len(points), 1), dtype=np.float32)
    homo = np.concatenate([points, ones], axis=1)
    return homo @ matrix.T


def align_face(frame, landmarks):
    landmarks = np.asarray(landmarks, dtype=np.float32)

    # 68-point landmark convention
    left_eye_center = landmarks[36:42].mean(axis=0)
    right_eye_center = landmarks[42:48].mean(axis=0)
    mouth_center = landmarks[48:68].mean(axis=0)

    src = np.float32([
        left_eye_center,
        right_eye_center,
        mouth_center
    ])

    dst = np.float32([
        DST_LEFT_EYE,
        DST_RIGHT_EYE,
        DST_MOUTH
    ])

    matrix = cv2.getAffineTransform(src, dst)

    aligned = cv2.warpAffine(
        frame,
        matrix,
        (ALIGNED_SIZE, ALIGNED_SIZE),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101
    )

    aligned_landmarks = transform_points(
        landmarks,
        matrix
    )

    return aligned, aligned_landmarks, matrix


def crop_fixed_mouth(aligned_face, aligned_landmarks):
    mouth_points = aligned_landmarks[48:68]

    mouth_center = mouth_points.mean(axis=0)

    # ขยับลงเล็กน้อย เพื่อให้รวมคาง/การอ้าปากโดยไม่ตัดริมฝีปากล่าง
    cx = int(round(mouth_center[0]))
    cy = int(round(mouth_center[1] + 3))

    half = MOUTH_CROP_SIZE // 2

    x1 = cx - half
    y1 = cy - half
    x2 = x1 + MOUTH_CROP_SIZE
    y2 = y1 + MOUTH_CROP_SIZE

    # padding หาก ROI เกินขอบ
    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - ALIGNED_SIZE)
    pad_bottom = max(0, y2 - ALIGNED_SIZE)

    if any([pad_left, pad_top, pad_right, pad_bottom]):
        padded = cv2.copyMakeBorder(
            aligned_face,
            pad_top, pad_bottom,
            pad_left, pad_right,
            cv2.BORDER_REFLECT_101
        )
        x1 += pad_left
        x2 += pad_left
        y1 += pad_top
        y2 += pad_top
    else:
        padded = aligned_face

    mouth = padded[y1:y2, x1:x2]

    if mouth.size == 0:
        raise RuntimeError("Mouth crop ว่าง")

    mouth_96 = cv2.resize(
        mouth,
        (MOUTH_OUTPUT_SIZE, MOUTH_OUTPUT_SIZE),
        interpolation=cv2.INTER_AREA
    )

    return mouth_96, mouth_center


print("Functions ready")

## Cell 7 — ทดสอบ 1 เฟรมก่อน

Cell นี้จะ:
- อ่านเฟรมแรกที่มี detection
- ใช้ bbox ของทุก track ในเฟรมนั้น
- หา 68 landmarks
- แสดง original + landmarks
- แสดง mouth crop ของแต่ละ track

ถ้าผลไม่ดี **อย่าเพิ่งรันทั้ง 100 เฟรม**

In [ ]:
import matplotlib.pyplot as plt

test_frame_index = int(det_df["frame_index"].min())
test_rows = det_df[
    det_df["frame_index"] == test_frame_index
].copy()

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, test_frame_index)
ret, test_frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("อ่าน test frame ไม่ได้")

boxes = test_rows[
    ["x1", "y1", "x2", "y2"]
].to_numpy(dtype=np.float32)

# FaceAlignment รับ detected_faces ที่เรามีอยู่แล้ว
landmark_result = fa.get_landmarks_from_image(
    test_frame,
    detected_faces=boxes.tolist()
)

if landmark_result is None:
    raise RuntimeError("FAN หา landmarks ไม่ได้")

print("Frame:", test_frame_index)
print("Faces:", len(test_rows))
print("Landmark sets:", len(landmark_result))

preview = test_frame.copy()
mouth_samples = []

for (_, row), landmarks in zip(
    test_rows.iterrows(),
    landmark_result
):
    track_id = int(row["track_id"])
    x1, y1, x2, y2 = map(
        int,
        [row["x1"], row["y1"], row["x2"], row["y2"]]
    )

    cv2.rectangle(
        preview,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    for x, y in np.asarray(landmarks).astype(int):
        cv2.circle(
            preview,
            (x, y),
            2,
            (0, 0, 255),
            -1
        )

    aligned, aligned_lms, _ = align_face(
        test_frame,
        landmarks
    )

    mouth_96, _ = crop_fixed_mouth(
        aligned,
        aligned_lms
    )

    mouth_samples.append(
        (track_id, mouth_96)
    )

preview_rgb = cv2.cvtColor(
    preview,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(15, 8))
plt.imshow(preview_rgb)
plt.title(f"Frame {test_frame_index} — 68 landmarks")
plt.axis("off")
plt.show()

for track_id, mouth_img in mouth_samples:
    plt.figure(figsize=(3, 3))
    plt.imshow(
        cv2.cvtColor(
            mouth_img,
            cv2.COLOR_BGR2RGB
        )
    )
    plt.title(f"Track {track_id} — Mouth 96x96")
    plt.axis("off")
    plt.show()

## Cell 8 — ตั้งค่าการประมวลผล

เริ่มจาก detection ที่มีอยู่ตอนนี้ก่อน (ประมาณ 100 เฟรม)  
เมื่อ QA ผ่านแล้วค่อยรัน Face Detection ทั้งคลิป แล้ว Notebook นี้จะประมวลผลตาม CSV ที่มี

In [ ]:
SAVE_ALIGNED_FACE = False
SAVE_PREVIEW_EVERY = 10

# quality thresholds เบื้องต้น
MIN_FACE_SCORE = 0.30
MIN_BLUR_SCORE = 20.0

print("SAVE_ALIGNED_FACE :", SAVE_ALIGNED_FACE)
print("MIN_FACE_SCORE    :", MIN_FACE_SCORE)
print("MIN_BLUR_SCORE    :", MIN_BLUR_SCORE)

## Cell 9 — Quality Metrics

In [ ]:
def calculate_blur_score(image):
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    return float(
        cv2.Laplacian(
            gray,
            cv2.CV_64F
        ).var()
    )


def calculate_brightness(image):
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    return float(gray.mean())


print("Quality functions ready")

## Cell 10 — Process detection.csv → Mouth Crops

Output แยกตาม `track_id`

ตัวอย่าง:

`dataset/mouth_crops/video001/track_000/frame_00000000.jpg`

In [ ]:
from collections import defaultdict

# สร้าง output track folders
track_ids = sorted(
    det_df["track_id"].astype(int).unique()
)

for tid in track_ids:
    (OUTPUT_ROOT / f"track_{tid:03d}").mkdir(
        parents=True,
        exist_ok=True
    )

# map frame -> rows
frame_groups = {
    int(frame_idx): group.copy()
    for frame_idx, group in det_df.groupby("frame_index")
}

target_frames = sorted(frame_groups.keys())

if not target_frames:
    raise RuntimeError("ไม่มี frame ใน detection.csv")

last_target_frame = max(target_frames)

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(f"เปิดวิดีโอไม่ได้: {VIDEO_PATH}")

fps = float(cap.get(cv2.CAP_PROP_FPS))

metadata_rows = []

frame_index = 0
processed_target_frames = 0
saved_mouths = 0
rejected_mouths = 0

while frame_index <= last_target_frame:
    ret, frame = cap.read()

    if not ret:
        break

    if frame_index not in frame_groups:
        frame_index += 1
        continue

    frame_rows = frame_groups[frame_index].sort_values(
        "track_id"
    )

    # filter low face confidence
    frame_rows = frame_rows[
        frame_rows["det_score"] >= MIN_FACE_SCORE
    ]

    if len(frame_rows) == 0:
        frame_index += 1
        continue

    boxes = frame_rows[
        ["x1", "y1", "x2", "y2"]
    ].to_numpy(dtype=np.float32)

    try:
        landmark_sets = fa.get_landmarks_from_image(
            frame,
            detected_faces=boxes.tolist()
        )
    except Exception as e:
        print(
            f"[WARN] frame {frame_index}: "
            f"landmark error: {e}"
        )
        frame_index += 1
        continue

    if landmark_sets is None:
        print(
            f"[WARN] frame {frame_index}: "
            "no landmarks"
        )
        frame_index += 1
        continue

    preview = frame.copy()

    # ป้องกัน length mismatch
    usable_count = min(
        len(frame_rows),
        len(landmark_sets)
    )

    for pos in range(usable_count):
        row = frame_rows.iloc[pos]
        landmarks = landmark_sets[pos]

        track_id = int(row["track_id"])
        det_score = float(row["det_score"])

        try:
            aligned, aligned_lms, matrix = align_face(
                frame,
                landmarks
            )

            mouth_96, mouth_center = crop_fixed_mouth(
                aligned,
                aligned_lms
            )

            blur_score = calculate_blur_score(
                mouth_96
            )

            brightness = calculate_brightness(
                mouth_96
            )

            accepted = blur_score >= MIN_BLUR_SCORE

            track_dir = (
                OUTPUT_ROOT
                / f"track_{track_id:03d}"
            )

            filename = (
                f"frame_{frame_index:08d}.jpg"
            )

            mouth_path = track_dir / filename

            if accepted:
                cv2.imwrite(
                    str(mouth_path),
                    mouth_96,
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
                saved_mouths += 1
            else:
                rejected_mouths += 1

            if SAVE_ALIGNED_FACE:
                aligned_dir = (
                    OUTPUT_ROOT
                    / f"track_{track_id:03d}"
                    / "aligned_faces"
                )
                aligned_dir.mkdir(
                    parents=True,
                    exist_ok=True
                )
                cv2.imwrite(
                    str(aligned_dir / filename),
                    aligned
                )

            timestamp = (
                frame_index / fps
                if fps > 0
                else 0.0
            )

            metadata_rows.append({
                "video_id": VIDEO_ID,
                "frame_index": frame_index,
                "timestamp_seconds": round(timestamp, 6),
                "track_id": track_id,
                "det_score": round(det_score, 6),
                "mouth_file": str(mouth_path.relative_to(PROJECT_ROOT))
                    if accepted else "",
                "blur_score": round(blur_score, 3),
                "brightness": round(brightness, 3),
                "accepted": bool(accepted),
            })

            # QA preview: bbox + mouth landmarks
            x1, y1, x2, y2 = map(
                int,
                [row["x1"], row["y1"], row["x2"], row["y2"]]
            )

            cv2.rectangle(
                preview,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            for x, y in np.asarray(
                landmarks[48:68]
            ).astype(int):
                cv2.circle(
                    preview,
                    (x, y),
                    3,
                    (0, 0, 255),
                    -1
                )

            cv2.putText(
                preview,
                f"ID {track_id}",
                (x1, max(25, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )

        except Exception as e:
            rejected_mouths += 1

            print(
                f"[WARN] frame={frame_index} "
                f"track={track_id}: {e}"
            )

    if (
        SAVE_PREVIEW_EVERY > 0
        and processed_target_frames % SAVE_PREVIEW_EVERY == 0
    ):
        cv2.imwrite(
            str(
                PREVIEW_DIR
                / f"frame_{frame_index:08d}.jpg"
            ),
            preview
        )

    processed_target_frames += 1

    if processed_target_frames % 10 == 0:
        print(
            f"Processed target frames: "
            f"{processed_target_frames}/{len(target_frames)} | "
            f"saved mouths={saved_mouths} | "
            f"rejected={rejected_mouths}"
        )

    frame_index += 1

cap.release()

mouth_meta_df = pd.DataFrame(
    metadata_rows
)

metadata_path = (
    OUTPUT_ROOT
    / "mouth_metadata.csv"
)

mouth_meta_df.to_csv(
    metadata_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print("Target frames :", len(target_frames))
print("Processed     :", processed_target_frames)
print("Saved mouths  :", saved_mouths)
print("Rejected      :", rejected_mouths)
print("Metadata      :", metadata_path)
print("Output        :", OUTPUT_ROOT)

## Cell 11 — ตรวจสถิติ Mouth Dataset

In [ ]:
if len(mouth_meta_df) == 0:
    print("ไม่มี mouth metadata")
else:
    summary = (
        mouth_meta_df
        .groupby("track_id")
        .agg(
            total=("frame_index", "count"),
            accepted=("accepted", "sum"),
            mean_blur=("blur_score", "mean"),
            mean_brightness=("brightness", "mean"),
            mean_det_score=("det_score", "mean"),
        )
    )

    summary["accept_rate"] = (
        summary["accepted"]
        / summary["total"]
    )

    display(summary)

## Cell 12 — ดู Mouth Crops ของแต่ละ Track

In [ ]:
from PIL import Image
from IPython.display import display

for tid in track_ids:
    track_dir = (
        OUTPUT_ROOT
        / f"track_{tid:03d}"
    )

    files = sorted(
        track_dir.glob("frame_*.jpg")
    )

    print(
        f"\nTrack {tid}: "
        f"{len(files)} mouth frames"
    )

    for path in files[:8]:
        print(path.name)
        display(
            Image.open(path).resize(
                (192, 192)
            )
        )

## Cell 13 — ดู QA Preview

In [ ]:
preview_files = sorted(
    PREVIEW_DIR.glob("*.jpg")
)

print("Preview files:", len(preview_files))

for path in preview_files[:10]:
    print(path.name)
    display(
        Image.open(path).resize(
            (960, 540)
        )
    )

# หลัง Notebook นี้ผ่าน

ผลที่ควรได้:

```text
dataset/mouth_crops/video001/
├── track_000/
│   ├── frame_00000000.jpg
│   ├── frame_00000001.jpg
│   └── ...
├── track_001/
│   ├── frame_00000000.jpg
│   ├── frame_00000001.jpg
│   └── ...
├── preview/
└── mouth_metadata.csv
```

## ขั้นต่อไป
1. ตรวจว่า Track ID ไม่สลับคน
2. ตรวจ mouth crop ว่าอยู่ตรงกลางและ scale คงที่
3. รัน Face Detection ทั้งวิดีโอ
4. รัน Mouth Extraction ทั้งวิดีโอ
5. สร้าง lip sequences
6. ทำ transcript/label alignment
7. แบ่ง train / validation / test
8. เริ่ม train baseline VSR